# Proyecto Final — Minería de Datos
## Clasificación de Diabetes (Pima Indians Dataset)

**Flujo de trabajo:**
1. Importación de librerías
2. Carga de datos
3. Exploración inicial (EDA)
4. Preprocesamiento
5. Modelado
6. Evaluación y comparación de modelos
7. Conclusiones

---
## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

---
## 2. Carga de datos

In [ ]:
df_train = pd.read_csv('Training.csv')
df_test  = pd.read_csv('Testing.csv')

print(f'Training : {df_train.shape[0]} filas x {df_train.shape[1]} columnas')
print(f'Testing  : {df_test.shape[0]}  filas x {df_test.shape[1]}  columnas')
df_train.head()

---
## 3. Exploración inicial (EDA)

In [ ]:
# Tipos de datos y nulos reales
df_train.info()

In [ ]:
# Estadísticas descriptivas
df_train.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Distribución de la variable objetivo
conteo = df_train['Outcome'].value_counts()
labels = ['Sin Diabetes (0)', 'Con Diabetes (1)']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, conteo, color=['steelblue', 'tomato'])
axes[0].set_title('Conteo por clase')
axes[0].set_ylabel('Cantidad')

axes[1].pie(conteo, labels=labels, autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Proporción por clase')

plt.suptitle('Variable Objetivo — Outcome', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de cada variable por clase
features = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
            'Insulin','BMI','DiabetesPedigreeFunction','Age']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flatten(), features):
    for val, color, label in [(0,'steelblue','Sin Diabetes'), (1,'tomato','Con Diabetes')]:
        ax.hist(df_train[df_train['Outcome']==val][col], bins=25,
                alpha=0.6, color=color, label=label, density=True)
    ax.set_title(col)
    ax.legend(fontsize=7)

plt.suptitle('Distribución de variables por clase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de correlación
plt.figure(figsize=(9, 7))
sns.heatmap(df_train.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Mapa de Correlación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Preprocesamiento

### 4.1 Tratamiento de valores faltantes
Las columnas `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` y `BMI` contienen **ceros biológicamente imposibles** que representan datos faltantes. Se reemplazan por `NaN` y se imputan con la **mediana por clase**.

In [ ]:
cols_con_ceros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Reporte de ceros antes de imputar
ceros = {col: (df_train[col] == 0).sum() for col in cols_con_ceros}
print('Ceros detectados (Training):')
for col, n in ceros.items():
    print(f'  {col:25s}: {n:4d}  ({n/len(df_train)*100:.1f}%)')

In [ ]:
def reemplazar_ceros_por_nan(df, columnas):
    df = df.copy()
    df[columnas] = df[columnas].replace(0, np.nan)
    return df

def imputar_por_mediana_clase(df, columnas, target='Outcome'):
    """Imputa NaN con la mediana de cada clase (0/1) para reducir sesgo."""
    df = df.copy()
    for col in columnas:
        mediana_clase = df.groupby(target)[col].transform('median')
        df[col] = df[col].fillna(mediana_clase)
    return df

df_train_clean = reemplazar_ceros_por_nan(df_train, cols_con_ceros)
df_train_clean = imputar_por_mediana_clase(df_train_clean, cols_con_ceros)

df_test_clean  = reemplazar_ceros_por_nan(df_test, cols_con_ceros)
# Para test: imputar con mediana global de train (evitar data leakage)
for col in cols_con_ceros:
    mediana_train = df_train_clean[col].median()
    df_test_clean[col] = df_test_clean[col].fillna(mediana_train)

print('Nulos restantes en train:', df_train_clean.isnull().sum().sum())
print('Nulos restantes en test :', df_test_clean.isnull().sum().sum())

### 4.2 Detección y tratamiento de outliers
Se usa el criterio **IQR** para detectar outliers. Se aplica **capping** (recorte) en lugar de eliminación para no perder datos.

In [ ]:
# Boxplots antes del capping
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flatten(), features):
    ax.boxplot(df_train_clean[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(col)

plt.suptitle('Boxplots — Detección de Outliers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def aplicar_capping_iqr(df, columnas, factor=1.5):
    """Recorta outliers al límite inferior/superior del IQR."""
    df = df.copy()
    for col in columnas:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        lower, upper = Q1 - factor * IQR, Q3 + factor * IQR
        df[col] = df[col].clip(lower=lower, upper=upper)
    return df

df_train_clean = aplicar_capping_iqr(df_train_clean, features)
df_test_clean  = aplicar_capping_iqr(df_test_clean, features)

print('Capping IQR aplicado correctamente.')

### 4.3 Separación de variables y estandarización

In [ ]:
X_train = df_train_clean[features]
y_train = df_train_clean['Outcome']

X_test  = df_test_clean[features]
y_test  = df_test_clean['Outcome']

scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit+transform solo en train
X_test_sc  = scaler.transform(X_test)        # solo transform en test

print('X_train:', X_train_sc.shape, '| X_test:', X_test_sc.shape)
print('y_train:', y_train.shape,    '| y_test:', y_test.shape)

---
## 5. Modelado

Se entrenan cuatro modelos y se comparan sus métricas.

In [ ]:
modelos = {
    'Regresión Logística' : LogisticRegression(max_iter=1000, random_state=42),
    'Árbol de Decisión'   : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42),
    'KNN (k=7)'           : KNeighborsClassifier(n_neighbors=7),
}

resultados = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train_sc, y_train)
    y_pred = modelo.predict(X_test_sc)
    y_prob = modelo.predict_proba(X_test_sc)[:, 1]

    resultados.append({
        'Modelo'   : nombre,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall'   : recall_score(y_test, y_pred),
        'F1-Score' : f1_score(y_test, y_pred),
        'ROC-AUC'  : roc_auc_score(y_test, y_prob),
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
df_resultados.round(4)

---
## 6. Evaluación y comparación de modelos

In [ ]:
# Gráfico comparativo de métricas
metricas = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
df_plot  = df_resultados[metricas]

x = np.arange(len(metricas))
width = 0.18
colors = ['steelblue', 'seagreen', 'tomato', 'goldenrod']

fig, ax = plt.subplots(figsize=(13, 5))
for i, (nombre, row) in enumerate(df_plot.iterrows()):
    ax.bar(x + i * width, row.values, width, label=nombre, color=colors[i], alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metricas)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Valor')
ax.set_title('Comparación de Métricas por Modelo', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, label='Umbral 0.80')
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (nombre, modelo) in zip(axes, modelos.items()):
    y_pred = modelo.predict(X_test_sc)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Diabetes', 'Diabetes'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nombre, fontsize=10)

plt.suptitle('Matrices de Confusión', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC
plt.figure(figsize=(7, 6))
for (nombre, modelo), color in zip(modelos.items(), colors):
    y_prob = modelo.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{nombre} (AUC={auc:.3f})', color=color, linewidth=2)

plt.plot([0,1],[0,1],'k--', linewidth=1)
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curvas ROC', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de características — Random Forest
rf = modelos['Random Forest']
importancias = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
importancias.plot(kind='bar', color='steelblue', alpha=0.85)
plt.title('Importancia de Variables — Random Forest', fontsize=12, fontweight='bold')
plt.ylabel('Importancia')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
print(importancias.to_string())

In [ ]:
# Mejor modelo según ROC-AUC
mejor = df_resultados['ROC-AUC'].idxmax()
print(f'Mejor modelo (ROC-AUC): {mejor}')
print(df_resultados.loc[mejor].to_string())

---
## 7. Conclusiones

| Aspecto | Resultado |
|---|---|
| **Dataset** | 2 460 muestras de entrenamiento, 9 variables clínicas, clasificación binaria (diabetes / no diabetes) |
| **Desbalance** | ~61% sin diabetes vs ~39% con diabetes — moderadamente desbalanceado |
| **Variables más importantes** | `Glucose`, `BMI` e `Insulin` tienen mayor poder discriminatorio |
| **Mejor modelo** | Determinado por la celda anterior según ROC-AUC |
| **Preprocesamiento clave** | Imputación de ceros biológicamente imposibles por mediana de clase + estandarización |

**Observaciones:**
- El nivel de glucosa es el predictor más relevante, seguido del IMC.
- La insulina presenta el mayor porcentaje de datos faltantes (~48%), lo que puede afectar su poder predictivo.
- Para mejorar el Recall en la clase positiva (diabetes), se podría aplicar SMOTE u oversampling.
- Random Forest es robusto ante outliers y variables correlacionadas, siendo generalmente el modelo con mejor desempeño global.